# Colab SSH via bore — Zero Signup
Run both cells, then paste the `bore.pub:PORT` output back into Windsurf.

In [ ]:
# Cell 1: Install SSH server + bore, set RANDOM password, start tunnel
# SECURITY: Password is now randomly generated each session.
# Previous hardcoded password 'colab1234' was ROTATED — do NOT reuse.
import subprocess, time, os, secrets, string

# Install SSH
subprocess.run(['apt-get', 'update', '-qq'], capture_output=True)
subprocess.run(['apt-get', 'install', '-y', 'openssh-server'], capture_output=True)
os.system('mkdir -p /run/sshd')

# Generate random password (ROTATED from leaked 'colab1234')
_password = ''.join(secrets.choice(string.ascii_letters + string.digits + '!@#$%') for _ in range(24))
os.system(f'echo "root:{_password}" | chpasswd')
os.system('sed -i "s/#PermitRootLogin.*/PermitRootLogin yes/" /etc/ssh/sshd_config')
os.system('sed -i "s/#PasswordAuthentication.*/PasswordAuthentication yes/" /etc/ssh/sshd_config')
os.system('service ssh start')
print('SSH server started.')
print(f'ROTATED PASSWORD (copy now, not stored anywhere): {_password}')

# Install bore
os.system('wget -qO /tmp/bore.tar.gz https://github.com/ekzhang/bore/releases/download/v0.5.0/bore-v0.5.0-x86_64-unknown-linux-musl.tar.gz')
os.system('tar -xzf /tmp/bore.tar.gz -C /usr/local/bin/')
print('bore installed')

# Start bore tunnel in background
proc = subprocess.Popen(
    ['bore', 'local', '22', '--to', 'bore.pub'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
time.sleep(5)

# Read output to find the port
output = ''
while True:
    line = proc.stdout.readline()
    if not line:
        break
    output += line
    if 'listening' in line.lower() or 'port' in line.lower():
        break
    if 'connected' in line.lower():
        break

print('\n=== BORE OUTPUT ===')
print(output)
print('=== END ===')
print('\nLook for a line like: listening at bore.pub:PORT')
print('Then on your Mac run: ssh root@bore.pub -p PORT')
print(f'Password: (the rotated password printed above — NOT colab1234)')

In [ ]:
# Cell 2: Keep tunnel alive (run this after connecting)
import time
while True:
    time.sleep(60)
    print('tunnel alive...', flush=True)

## Connect from Windsurf
Paste the `bore.pub:PORT` line here and I'll SSH in directly.